<a href="https://colab.research.google.com/github/ThandiweM/msc-ai-thesis/blob/feature%2Fed5005-data-imbalance/ed5005_etivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student name: Thandiwe Feziwe Mangana
# Student ID: 24325104

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install polars
!pip install fg-data-profiling

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, auc, accuracy_score, precision_recall_fscore_support, precision_recall_curve, average_precision_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score
from sklearn import set_config
import pickle
import matplotlib.pyplot as plt
%matplotlib inline
from scipy.stats import loguniform, randint, uniform
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import ExtraTreesClassifier
pd.set_option('display.max_columns', None)
from pathlib import Path
from data_profiling import ProfileReport
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.model_selection import learning_curve
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import shap
from sklearn.model_selection import StratifiedGroupKFold
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
import time

##Utilities

In [ ]:
class Protocol:

    SCORING = {
        'pr_auc': 'average_precision',
        'f1': make_scorer(f1_score, pos_label=1),
        'precision': make_scorer(precision_score, pos_label=1),
        'recall': make_scorer(recall_score, pos_label=1),}

    SETTINGS = dict(cv_strategy = StratifiedGroupKFold(n_splits=5, shuffle=True,random_state=42), scoring = SCORING , n_iter = 20, random_state = 42)
    V_SETTINGS_RANDOM = dict(cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring = SCORING, n_iter = 20, random_state = 42)

    MODEL_COLORS = {
    'Random Forest': 'steelblue',
    'XGBoost': 'darkorange',
    'LightGBM': 'seagreen',}


In [ ]:
class ResultsUtils:

    @staticmethod
    def plot_class_distribution(y, level_name, save_path):
      # ensures 0 comes before 1
      counts = pd.Series(y).value_counts().sort_index()
      labels = ['Legitimate', 'Fraud']

      plt.figure(figsize=(7, 6))
      bars = plt.bar(labels, counts.values, color=['steelblue', 'firebrick'])

      # Adding the actual count and percentage on top of each bar
      total = counts.sum()
      for bar, count in zip(bars, counts.values):
          pct = count / total * 100
          plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{count}\n({pct:.1f}%)', ha='center', va='bottom')

      plt.ylim(0, counts.max() * 1.15)
      plt.title(f'Class Distribution — {level_name}')
      plt.ylabel('Count')
      plt.tight_layout()
      plt.savefig(save_path, dpi=150, bbox_inches='tight')
      plt.show()

    @staticmethod
    def plot_wallet_activity_histogram(X, wallet_col, save_path):
      counts = X.groupby(wallet_col).size()
      plt.figure(figsize=(10, 6))
      sns.histplot(counts, bins=50, log_scale=(False, False))
      plt.title("Distribution of Transactions per Wallet")
      plt.xlabel("Transactions per Wallet")
      plt.ylabel("Count (log scale)")
      plt.tight_layout()
      plt.savefig(save_path, dpi=150, bbox_inches='tight')
      plt.show()

    @staticmethod
    def get_wallet_structure_summary(X, wallet_col='address'):
      counts = X.groupby(wallet_col).size()
      row = {
          'Unique Wallets': X[wallet_col].nunique(),
          'Mean Rows Per Wallet': counts.mean(),
          'Median Rows per Wallet': counts.median(),
          'Max Rows per Wallet': counts.max(),
      }

      return pd.DataFrame([row])

    @staticmethod
    def plot_correlation_heatmap(X, save_path, title="Feature Correlation Heatmap"):

        corr_matrix = X.corr()

        plt.figure(figsize=(16, 14))

        # Create the heatmap using seaborn
        # cmap chooses the color palette
        sns.heatmap(
            corr_matrix,
            annot=False,
            cmap="coolwarm",
            vmin=-1,
            vmax=1,
            square=True,
            linewidths=0.1,
            cbar_kws={'label': 'Correlation Coefficient'}
        )

        # Add title and layout adjustments
        plt.title(title, fontsize=14, pad=15)
        plt.xticks(rotation=90, ha='right', fontsize=6)
        plt.yticks(rotation=0, fontsize=6)
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

    @staticmethod
    def plot_feature_boxplot(X, y, feature, save_path, log_scale = False):

        labels = ['Legitimate', 'Fraud']

        plot_data = X[feature]
        ylabel = feature
        if log_scale:
            plot_data = np.log1p(plot_data)
            ylabel = f'{feature} (log-transformed)'

        plt.figure(figsize=(7, 6))
        ax = sns.boxplot(x=y, y=plot_data, palette=['steelblue', 'firebrick'])
        ax.set_xticklabels(labels)
        plt.title(f'Distribution of {feature} by Outcome')
        plt.xlabel('Class')
        plt.ylabel(ylabel)
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

    @staticmethod
    def get_wallet_level_leakage_check(X, wallet_cols, address_col='address'):
        X = X.rename(columns = FeatureTaxanomy.FEATURE_MAP)
        n_fingerprints = X[wallet_cols].drop_duplicates().shape[0]
        n_addresses = X[address_col].nunique()

        row = {
            'Unique Wallet Feature Combinations': n_fingerprints,
            'Unique Addresses': n_addresses,
        }
        return pd.DataFrame([row])

    @staticmethod
    def get_cv_summary(search, model_name):

        best_index = search.best_index_
        row = {'Model': model_name}

        display_names = {'pr_auc': 'PR-AUC', 'f1': 'F1 Score',
                          'precision': 'Precision', 'recall': 'Recall'}

        for metric_key in search.scoring:
            label = display_names.get(metric_key, metric_key)  # fallback
            mean = search.cv_results_[f'mean_test_{metric_key}'][best_index]
            std = search.cv_results_[f'std_test_{metric_key}'][best_index]
            row[f'CV {label} (mean)'] = mean
            row[f'CV {label} (std)'] = std

        return pd.DataFrame([row])

    @staticmethod
    def get_holdout_summary(estimator, X_test, y_test, model_name):
        y_true = y_test.values.ravel() if hasattr(y_test, 'values') else y_test
        model = estimator

        y_probs = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='binary', pos_label=1, zero_division=0
        )

        pr_auc = average_precision_score(y_true, y_probs)

        row = {
            'Model': model_name,
            'Holdout Precision': precision,
            'Holdout Recall': recall,
            'Holdout F1 Score': f1,
            'Holdout PR-AUC': pr_auc,
        }

        summary_df = pd.DataFrame([row])

        return summary_df, y_true, y_probs, y_pred

    def plot_confusion_matrix(y_true, y_pred, model_name, save_path):
      cm = confusion_matrix(y_true, y_pred)
      display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])

      fig, ax = plt.subplots(figsize=(6, 6))
      display.plot(ax=ax, cmap='Blues', values_format='d')

      ax.set_title(f'Confusion Matrix : {model_name}')
      plt.tight_layout()
      plt.savefig(save_path, dpi=150, bbox_inches='tight')
      plt.show()


    @staticmethod
    def plot_pr_curves_overlay(results_dict, title, save_path):
      fig, ax = plt.subplots(figsize=(8, 7))

      for model_name, (y_true, y_probs) in results_dict.items():

            precision, recall, _ = precision_recall_curve(y_true, y_probs)
            pr_auc = average_precision_score(y_true, y_probs)

            # falls back to matplotlib default if not found
            color = Protocol.MODEL_COLORS.get(model_name, None)

            ax.plot(recall, precision, label=f'{model_name} (PR-AUC = {pr_auc:.3f})',
                    linewidth=2, color=color)

      ax.set_xlabel('Recall')
      ax.set_ylabel('Precision')
      ax.set_title(f'Precision-Recall Curves : {title}')
      ax.legend(loc='lower left')
      ax.set_xlim([0, 1])
      ax.set_ylim([0, 1.05])
      plt.tight_layout()
      plt.savefig(save_path, dpi=150, bbox_inches='tight')
      plt.show()

    @staticmethod
    def plot_learning_curve(estimator, X_train, y_train, groups, cv_strategy, scoring, random_state, model_name, save_path):
        pipeline = estimator
        cv_strategy = cv_strategy
        train_sizes = np.linspace(0.1, 1.0, 8)
        model_color = Protocol.MODEL_COLORS.get(model_name, None)

        train_sizes, train_scores, val_scores = learning_curve(
              pipeline, X_train, y_train,
              groups = groups,
              cv = cv_strategy,
              n_jobs = -1,
              train_sizes = train_sizes,
              scoring =  scoring,
              shuffle = True,
              random_state =  random_state,
              error_score = 'raise',
          )

        train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
        val_mean,   val_std   = val_scores.mean(axis=1),   val_scores.std(axis=1)

        plt.figure(figsize=(10, 6))
        plt.plot(train_sizes, train_mean, 'o-', color=model_color, label='Training Score')
        plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                          alpha=0.15, color=model_color)

        plt.plot(train_sizes, val_mean, 'o--', color=model_color, alpha=0.6, label='Cross Validation Score')
        plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                          alpha=0.10, color=model_color)

        plt.title(f'Learning Curve — {model_name}')
        plt.xlabel('Training Set Size')
        plt.ylabel('PR AUC')
        plt.legend(loc='best')
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

    @staticmethod
    def plot_shap_beeswarm(shap_values, X, model_name, save_path):
      plt.figure(figsize=(10, 8))
      shap.summary_plot(shap_values, X, show=False)
      plt.title(f'SHAP Summary Plot : {model_name}')
      plt.tight_layout()
      plt.savefig(save_path, dpi=150, bbox_inches='tight')
      plt.show()

    @staticmethod
    def get_shap_importance_ranking(shap_values, X):
      #Calculate mean absolute SHAP value for every column
      importance_df = pd.DataFrame({
          'feature': X.columns,
          'importance': np.abs(shap_values).mean(0)  # Average per feaure (ie. take the average of all transaction rows for each column)
      }).sort_values(by='importance', ascending=False).reset_index(drop=True)
      return importance_df

    def plot_shap_importance_bar(model_name, importance_df, cutoff_index, save_path):
      plt.figure(figsize=(10, 6))
      plt.plot(importance_df.index, importance_df['importance'],
                marker='o', linestyle='-', markersize=4, color='steelblue')

      plt.axvline(x=cutoff_index, color='firebrick', linestyle='--',
                  label=f'Cutoff: top {cutoff_index} features retained')

      plt.title(f'Feature Importance Elbow Plot : {model_name}')
      plt.xlabel('Number of Features Included')
      plt.ylabel('Mean Absolute SHAP Value')
      plt.legend()
      plt.grid(True, alpha=0.3)
      plt.tight_layout()
      plt.savefig(save_path, dpi=150, bbox_inches='tight')
      plt.show()


    @staticmethod
    def combine_tables(tab_1, tab_2, tab_3):
      combined_table = pd.concat([tab_1, tab_2, tab_3], ignore_index=True)
      return combined_table

    @staticmethod
    def get_parity_check(full_pr_auc, full_pr_auc_std, pruned_pr_auc, config_label='Pruned'):

      tolerance = full_pr_auc_std
      drop = full_pr_auc - pruned_pr_auc
      accepted = drop <= tolerance

      row = {
          'Configuration': config_label,
          'Full PR-AUC': full_pr_auc,
          'Pruned PR-AUC': pruned_pr_auc,
          'Drop': drop,
          'Tolerance (Champion Std)': tolerance,
          'Accepted': accepted,
      }

      status = "ACCEPTED" if accepted else "REJECTED"

      return pd.DataFrame([row])

###Data loading and sampling

In [ ]:
# @title
class FeatureTaxanomy:
    """
    Contains features' schema

    """

    FEATURE_MAP = {
    'length_to':'length_to_address',
    'is_contract_creation':'coming_from_contract',
    'length_from':'length_from_address',
    'is_same_address':'from_is_same_as_to_address',
    'duration_seconds':'duration',
    'gas_used.gas_used_summation':'gas_used_summation',
    'gas_used.gas_used_average':'gas_used_average',
    'gas_used.gas_used_median':'gas_used_median',
    'gas_used.gas_used_standard_deviation':'gas_used_standard_deviation',
    'gas_used.gas_used_maximum_val':'gas_used_maximum_val',
    'gas_used.gas_used_minimum_val':'gas_used_minimum_val',
    'gas_used.gas_used_variance':'gas_used_variance',
    'gas_used.gas_used_range_value':'gas_used_range_value',
    'gas_used.gas_used_skewness':'gas_used_skewness',
    'gas_used.gas_used_mode':'gas_used_mode',
    'gas_used.gas_used_coefficient_of_variation':'gas_used_coefficient_of_variation',
    'gas_prices.gas_prices_summation':'gas_prices_summation',
    'gas_prices.gas_prices_average':'gas_prices_average',
    'gas_prices.gas_prices_median':'gas_prices_median',
    'gas_prices.gas_prices_standard_deviation':'gas_prices_standard_deviation',
    'gas_prices.gas_prices_maximum_val':'gas_prices_maximum_val',
    'gas_prices.gas_prices_minimum_val':'gas_prices_minimum_val',
    'gas_prices.gas_prices_variance':'gas_prices_variance',
    'gas_prices.gas_prices_range_value':'gas_prices_range_value',
    'gas_prices.gas_prices_skewness':'gas_prices_skewness',
    'gas_prices.gas_prices_mode':'gas_prices_mode',
    'gas_prices.gas_prices_coefficient_of_variation':'gas_prices_coefficient_of_variation',
    'cumulativeGasUsed.cumulativeGasUsed_summation':'cumulativeGasUsed_summation',
    'cumulativeGasUsed.cumulativeGasUsed_average':'cumulativeGasUsed_average',
    'cumulativeGasUsed.cumulativeGasUsed_median':'cumulativeGasUsed_median',
    'cumulativeGasUsed.cumulativeGasUsed_standard_deviation':'cumulativeGasUsed_standard_deviation',
    'cumulativeGasUsed.cumulativeGasUsed_maximum_val':'cumulativeGasUsed_maximum_val',
    'cumulativeGasUsed.cumulativeGasUsed_minimum_val':'cumulativeGasUsed_minimum_val',
    'cumulativeGasUsed.cumulativeGasUsed_variance':'cumulativeGasUsed_variance',
    'cumulativeGasUsed.cumulativeGasUsed_range_value':'cumulativeGasUsed_range_value',
    'cumulativeGasUsed.cumulativeGasUsed_skewness':'cumulativeGasUsed_skewness',
    'cumulativeGasUsed.cumulativeGasUsed_mode':'cumulativeGasUsed_mode',
    'cumulativeGasUsed.cumulativeGasUsed_coefficient_of_variation':'cumulativeGasUsed_coefficient_of_variation',
    'values.values_summation':'values_summation',
    'values.values_average':'values_average',
    'values.values_median':'values_median',
    'values.values_standard_deviation':'values_standard_deviation',
    'values.values_maximum_val':'values_maximum_val',
    'values.values_minimum_val':'values_minimum_val',
    'values.values_variance':'values_variance',
    'values.values_range_value':'values_range_value',
    'values.values_skewness':'values_skewness',
    'values.values_mode':'values_mode',
    'values.values_coefficient_of_variation':'values_coefficient_of_variation',
    'nonce.nonce_summation':'nonce_summation',
    'nonce.nonce_average':'nonce_average',
    'nonce.nonce_median':'nonce_median',
    'nonce.nonce_standard_deviation':'nonce_standard_deviation',
    'nonce.nonce_maximum_val':'nonce_maximum_val',
    'nonce.nonce_minimum_val':'nonce_minimum_val',
    'nonce.nonce_variance':'nonce_variance',
    'nonce.nonce_range_value':'nonce_range_value',
    'nonce.nonce_skewness':'nonce_skewness',
    'nonce.nonce_mode':'nonce_mode',
    'nonce.nonce_coefficient_of_variation':'nonce_coefficient_of_variation'}

    # Features kept focuse on transaction level and wallet level behavioural features.
    # The Token metadata features (ie. ERC-20/ERC-721 ) are excluded to avoid reintroducing
    # entity identifying columns
    # and to keep the feature space focused on transactional behaviour.
    FEATURE_TO_KEEP = [
    'length_transaction_hash',
    'length_to_address',
    'coming_from_contract',
    'status',
    'log_removed',
    'block_number',
    'gas_used',
    'length_from_address',
    'index',
    'gas_efficiency',
    'value',
    'chain_id',
    'message',
    'total_gas_cost',
    'gas_per_log_event',
    'log_index',
    'event_activity_flag',
    'normalized_token_transfer',
    'effective_gas_price',
    'cumulative_gas_used',
    'from_is_same_as_to_address',
    'gas_price_ratio',
    'token_transfer_amount',
    'length_log',
    'Error!',
    'log_count',
    'num_transaction',
    'duration',
    'number_of_errors',
    'error_rate',
    'gas_used_summation',
    'gas_used_average',
    'gas_used_median',
    'gas_used_standard_deviation',
    'gas_used_maximum_val',
    'gas_used_minimum_val',
    'gas_used_variance',
    'gas_used_range_value',
    'gas_used_skewness',
    'gas_used_mode',
    'gas_used_coefficient_of_variation',
    'gas_prices_summation',
    'gas_prices_average',
    'gas_prices_median',
    'gas_prices_standard_deviation',
    'gas_prices_maximum_val',
    'gas_prices_minimum_val',
    'gas_prices_variance',
    'gas_prices_range_value',
    'gas_prices_skewness',
    'gas_prices_mode',
    'gas_prices_coefficient_of_variation',
    'cumulativeGasUsed_summation',
    'cumulativeGasUsed_average',
    'cumulativeGasUsed_median',
    'cumulativeGasUsed_standard_deviation',
    'cumulativeGasUsed_maximum_val',
    'cumulativeGasUsed_minimum_val',
    'cumulativeGasUsed_variance',
    'cumulativeGasUsed_range_value',
    'cumulativeGasUsed_skewness',
    'cumulativeGasUsed_mode',
    'cumulativeGasUsed_coefficient_of_variation',
    'values_summation',
    'values_average',
    'values_median',
    'values_standard_deviation',
    'values_maximum_val',
    'values_minimum_val',
    'values_variance',
    'values_range_value',
    'values_skewness',
    'values_mode',
    'values_coefficient_of_variation',
    'nonce_summation',
    'nonce_average',
    'nonce_median',
    'nonce_standard_deviation',
    'nonce_maximum_val',
    'nonce_minimum_val',
    'nonce_variance',
    'nonce_range_value',
    'nonce_skewness',
    'nonce_mode',
    'nonce_coefficient_of_variation',
    'number_of_from_address',
    'number_of_unique_from_address',
    'number_of_to_address',
    'number_of_unique_to_address'
    ]


    FEATURES_WALLET = [
        'number_of_errors',
        'error_rate',
        'gas_used_summation',
        'gas_used_average',
        'gas_used_median',
        'gas_used_standard_deviation',
        'gas_used_maximum_val',
        'gas_used_minimum_val',
        'gas_used_variance',
        'gas_used_range_value',
        'gas_used_skewness',
        'gas_used_mode',
        'gas_used_coefficient_of_variation',
        'gas_prices_summation',
        'gas_prices_average',
        'gas_prices_median',
        'gas_prices_standard_deviation',
        'gas_prices_maximum_val',
        'gas_prices_minimum_val',
        'gas_prices_variance',
        'gas_prices_range_value',
        'gas_prices_skewness',
        'gas_prices_mode',
        'gas_prices_coefficient_of_variation',
        'cumulativeGasUsed_summation',
        'cumulativeGasUsed_average',
        'cumulativeGasUsed_median',
        'cumulativeGasUsed_standard_deviation',
        'cumulativeGasUsed_maximum_val',
        'cumulativeGasUsed_minimum_val',
        'cumulativeGasUsed_variance',
        'cumulativeGasUsed_range_value',
        'cumulativeGasUsed_skewness',
        'cumulativeGasUsed_mode',
        'cumulativeGasUsed_coefficient_of_variation',
        'values_summation',
        'values_average',
        'values_median',
        'values_standard_deviation',
        'values_maximum_val',
        'values_minimum_val',
        'values_variance',
        'values_range_value',
        'values_skewness',
        'values_mode',
        'values_coefficient_of_variation',
        'nonce_summation',
        'nonce_average',
        'nonce_median',
        'nonce_standard_deviation',
        'nonce_maximum_val',
        'nonce_minimum_val',
        'nonce_variance',
        'nonce_range_value',
        'nonce_skewness',
        'nonce_mode',
        'nonce_coefficient_of_variation',
        'number_of_from_address',
        'number_of_unique_from_address',
        'number_of_to_address',
        'number_of_unique_to_address',
        'num_transaction',
        'duration'
        ]

In [ ]:
# @title

class DataIngestion:
    """
    Contains functions for data ingestion and preprocessing.

    """

    base_path = ''
    fraud_path = ''
    legitimate_path = ''
    fraud_parquet_path = ''
    legitimate_parquet_path = ''

    def __init__(self, base_path, fraud_filename, legitimate_filename):

        """
        Constructor for the DataIngester class.
        It takes the parameters and generates file paths.

        Parameters
        ----------
        base_path : string
            The path to the folder in which all the data is stored.
        fraud_filename : string
            The name of the file containing the fraud data.
        legitimate_filename : string
            The name of the file containing the legitimate data.

        """
        self.base_path = base_path
        self.fraud_path = self.base_path + fraud_filename
        self.legitimate_path = self.base_path + legitimate_filename

        fraud_parquet_filename = 'fraud.parquet'
        self.fraud_parquet_path = self.base_path + fraud_parquet_filename
        legitimate_parquet_filename = 'legitimate.parquet'
        self.legitimate_parquet_path = self.base_path + legitimate_parquet_filename

    def ingest_data(self):

        """
        Ingests the raw data from the csv files and saves it as parquet files.

        This is to optimise for large csv files

        Parameters
        ----------
        None

        Returns
        -------
        tuple
            The paths to the generated and saved parquet files.

        """
        q_fraud = pl.scan_csv(self.fraud_path,
                       infer_schema_length=10000,
                       ignore_errors=True,
                       schema_overrides={'token_transfer_amount':pl.Float64})

        q_fraud.sink_parquet(self.fraud_parquet_path)

        q_legitimate = pl.scan_csv(self.legitimate_path,
                       infer_schema_length=10000,
                       ignore_errors=True,
                       schema_overrides={'token_transfer_amount':pl.Float64})

        q_legitimate.sink_parquet(self.legitimate_parquet_path)

        return self.fraud_parquet_path, self.legitimate_parquet_path

    def extract_subset(self, fraud_parquet_path, legitimate_parquet_path, n_samples=20000):

        """
        Extracts a subset of the data from the parquet files.
        It combines the fraud and legitimate data and saves it as a parquet file.

        Parameters
        ----------
        n_samples : int
            The number of samples to extract from each dataset.
            The default is 20000.
        fraud_parquet_path : string
            The path to the parquet file containing the fraud data.
        legitimate_parquet_path : string
            The path to the parquet file containing the legitimate data.

        Returns
        -------
        string
            The path to the generated and saved parquet file.

        """

        lazy_legitimate = pl.scan_parquet(legitimate_parquet_path)
        lazy_fraud = pl.scan_parquet(fraud_parquet_path)

        # Creating one dataset and extracting a sample of 20 000
        legitimate_count = lazy_legitimate.select(pl.len()).collect().item()
        fraud_count = lazy_fraud.select(pl.len()).collect().item()

        print(f"Normal transactions: {legitimate_count}")
        print(f"Fraud transactions: {fraud_count}")

        # Calculating the target for n_samples rows
        total_sample = n_samples
        fraud_ratio = fraud_count / (legitimate_count + fraud_count)
        target_fraud = int(total_sample * fraud_ratio)
        target_legitimate = total_sample - target_fraud

        # List of columns that have a mismatch
        mismatch_cols = ['erc_721_TokenAddress', 'erc_721_TokenName', 'erc_721_TokenSymbol']

        # Standardise both LazyFrames to String
        lazy_fraud = lazy_fraud.with_columns([
            pl.col(c).cast(pl.String) for c in mismatch_cols
        ])

        lazy_legitimate = lazy_legitimate.with_columns([
            pl.col(c).cast(pl.String) for c in mismatch_cols
        ])

        # Sampling from LazyFrames
        df_fraud = lazy_fraud.collect().sample(n=target_fraud, shuffle=True, seed=42)
        df_legitimate = lazy_legitimate.collect().sample(n=target_legitimate, shuffle=True, seed=42)

        # Concatenating and shuffling
        df_final = pl.concat([df_fraud, df_legitimate]).sample(fraction=1.0, shuffle=True, seed=42)

        # Save as parquet
        df_final.write_parquet(self.base_path + 'dataset.parquet')

        # Save as csv
        df_final.write_csv(self.base_path + 'dataset.csv')
        return self.base_path + 'dataset.parquet'

    from sklearn.model_selection import StratifiedGroupKFold

    def grouped_train_test_split(self, X, y, groups, n_splits=5, random_state=42):
      sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                random_state=random_state)
      train_idx, test_idx = next(sgkf.split(X, y, groups))

      return (X.iloc[train_idx], X.iloc[test_idx],
            y.iloc[train_idx], y.iloc[test_idx],
            groups.iloc[train_idx], groups.iloc[test_idx])


    def get_processed_data(self, dataset_parquet_path, grouped_split = False):

        """
        Loads the dataset from the parquet file and splits it into training and testing sets.
        It also renames the columns to more align with the feature taxonomy.

        Parameters
        ----------
        string: dataset_parquet_path
            The path to the parquet file containing the dataset.

        Returns
        -------
        tuple
            The X_train, X_test, y_train, y_test.

        """

        df = pd.read_parquet(dataset_parquet_path)
        y = df['flag']
        X = df.drop(columns=['flag'], axis=1)
        groups = df.loc[X.index, "address"]
        groups_train = pd.Series(dtype="str")
        groups_test = pd.Series(dtype="str")

        self.y = y
        self.X = X
        self.groups = groups
        self.wallet_labels = df.groupby('address')['flag'].first()

        if grouped_split:
          X_train, X_test, y_train, y_test, groups_train, groups_test = self.grouped_train_test_split(X, y, groups)
        else:
          X_train, X_test, y_train, y_test= train_test_split(
          X,
          y,
          test_size=0.2,
          stratify=y,
          random_state=42
          )

        X_train = X_train.rename(columns=FeatureTaxanomy.FEATURE_MAP)
        X_test = X_test.rename(columns=FeatureTaxanomy.FEATURE_MAP)

        X_train = X_train[FeatureTaxanomy.FEATURE_TO_KEEP]
        X_test = X_test[FeatureTaxanomy.FEATURE_TO_KEEP]

        return X_train, X_test, y_train, y_test, groups_train, groups_test


In [ ]:
legitimate_parquet_path = "" #= '/content/drive/MyDrive/thesis_data/DeFiTransLyzer_legitimate.parquet'
fraud_parquet_path  = "" #= "/content/drive/MyDrive/thesis_data/DeFiTransLyzer_fraud.parquet"
dataset_path = '/content/drive/MyDrive/thesis_data/dataset.parquet'
eda_results_path = '/content/drive/MyDrive/thesis_data/Results/EDA/'
experiments_results_path = '/content/drive/MyDrive/thesis_data/Results/Experiments/'

data_ingestion = DataIngestion('/content/drive/MyDrive/thesis_data/', 'DeFiTransLyzer_fraud.csv', 'DeFiTransLyzer_legitimate.csv')
data_ingestion.get_processed_data(dataset_path, grouped_split=True)
#fraud_parquet_path, legitimate_parquet_path = data_ingestion.ingest_data()
#dataset_path = data_ingestion.extract_subset(fraud_parquet_path, legitimate_parquet_path, n_samples = 50000)


In [ ]:
ResultsUtils.plot_class_distribution(data_ingestion.wallet_labels,'Wallet level', eda_results_path+'class_distribution_wallet_level.png')

In [ ]:
ResultsUtils.plot_class_distribution(data_ingestion.y,'Transaction level',  eda_results_path+'class_distribution_transaction_level.png')

In [ ]:
wallet_summary = ResultsUtils.get_wallet_structure_summary(data_ingestion.X,'address')
wallet_summary.to_csv(eda_results_path+'wallet_structure.csv', index=False)

In [ ]:
ResultsUtils.plot_wallet_activity_histogram(data_ingestion.X, 'address', eda_results_path+'wallet_activity_histogrm.png')

###Exploratory data analysis (EDA) and data preparation

In [ ]:
# @title

class DataPreparation:

    SKEWED_FEATURES = ['values_minimum_val','total_gas_cost','gas_used_minimum_val','value','nonce_minimum_val','values_variance']
    CATEGORICAL_FEATURES = ['event_activity_flag','chain_id','from_is_same_as_to_address']
    STANDARD_FEATURES =  ['length_transaction_hash','coming_from_contract','status','log_removed','block_number',
                          'gas_used', 'length_from_address','index','gas_efficiency', 'message','gas_per_log_event','log_index','normalized_token_transfer','effective_gas_price','cumulative_gas_used','gas_price_ratio',
        'token_transfer_amount','length_log','Error!','log_count','num_transaction','duration','number_of_errors','error_rate','gas_used_summation','gas_used_average','gas_used_median','gas_used_standard_deviation','gas_used_maximum_val','gas_used_variance','gas_used_range_value',
        'gas_used_skewness', 'gas_used_mode','gas_used_coefficient_of_variation','gas_prices_summation', 'gas_prices_average', 'gas_prices_median','gas_prices_standard_deviation','gas_prices_maximum_val','gas_prices_minimum_val','gas_prices_variance', 'gas_prices_range_value',
        'gas_prices_skewness','gas_prices_mode','gas_prices_coefficient_of_variation', 'cumulativeGasUsed_summation','cumulativeGasUsed_average','cumulativeGasUsed_median','cumulativeGasUsed_standard_deviation', 'cumulativeGasUsed_maximum_val', 'cumulativeGasUsed_minimum_val', 'cumulativeGasUsed_variance',
        'cumulativeGasUsed_range_value', 'cumulativeGasUsed_skewness','cumulativeGasUsed_mode','cumulativeGasUsed_coefficient_of_variation', 'values_summation', 'values_average','values_median', 'values_standard_deviation','values_maximum_val','values_range_value',
        'values_skewness','values_mode','values_coefficient_of_variation','nonce_summation','nonce_average','nonce_median', 'nonce_standard_deviation','nonce_maximum_val','nonce_variance','nonce_range_value', 'nonce_skewness','nonce_mode', 'nonce_coefficient_of_variation',
        'number_of_from_address','number_of_unique_from_address','number_of_to_address','number_of_unique_to_address' ]
    BINARIZE_FEATURES = ['length_to_address']


    @property
    def skewed_features(self):
      return list( set(FeatureTaxanomy.FEATURE_TO_KEEP).intersection(self.SKEWED_FEATURES))

    @property
    def standard_features(self):
      return list( set(FeatureTaxanomy.FEATURE_TO_KEEP).intersection(self.STANDARD_FEATURES))

    @property
    def categorical_features(self):
      return list( set(FeatureTaxanomy.FEATURE_TO_KEEP).intersection(self.CATEGORICAL_FEATURES))

    @property
    def binarize_features(self):
      return list( set(FeatureTaxanomy.FEATURE_TO_KEEP).intersection(self.BINARIZE_FEATURES))

    def update_features(self, features_to_drop):
      FeatureTaxanomy.FEATURE_TO_KEEP = list(set(FeatureTaxanomy.FEATURE_TO_KEEP) - set(features_to_drop))

    def plot_feature_by_fraud_flag(self, X_train, y_train, feature):
      # Check distribution of a feature by fraud status
      plt.figure(figsize=(10, 6))
      sns.boxplot(x=y_train, y=X_train[feature])
      plt.title(f'Distribution of {feature} by outcome')
      plt.show()

    # Initial profiling
    def generate_profiling_report(self, X_train, report_path):
      profile_data = X_train.drop(columns=['df_index'], errors='ignore')
      profile = ProfileReport(profile_data, title="Thesis Dataset Profiling: Full Sample",
                            correlations={
            "pearson": {"calculate": True},
            "spearman": {"calculate": True},
            "kendall": {"calculate": False},
            "phi_k": {"calculate": False},
        },
        interactions={"continuous": False},
        minimal=True)
      # Save the report as an HTML file
      profile.to_file(report_path)

    def generate_preprocessing_pipeline(self):

        clean_inf = FunctionTransformer(
            lambda X: np.where(np.isinf(X), np.nan, X),
            feature_names_out="one-to-one")

        enforce_finite = FunctionTransformer(
            lambda X: np.clip(np.nan_to_num(X, posinf=1e18, neginf=-1e18), -1e18, 1e18),
            feature_names_out="one-to-one")


        skew_pipeline = Pipeline(steps=[
        ('clean_inf', clean_inf),
        ('enforce_finite', enforce_finite),
        ('imputer', SimpleImputer(strategy='median')),
        ('log', FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ('scale', RobustScaler())])

        standard_pipeline = Pipeline(steps=[
        ('clean_inf', clean_inf),
        ('enforce_finite', enforce_finite),
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())])

        binarize_cat = FunctionTransformer(
            lambda X: (X == 0).astype(int),
            feature_names_out="one-to-one")

        binary_pipe = Pipeline(steps=[
            ('binarize', binarize_cat)])

        flag_pipeline = Pipeline(steps=[
        ('passthrough', FunctionTransformer(lambda X: X))])

        preprocess_pipeline = ColumnTransformer(
        transformers=[
            ('skew', skew_pipeline, self.skewed_features),
            ('standard', standard_pipeline, self.standard_features),
            ('binary', binary_pipe, self.binarize_features),
            ('flags', flag_pipeline, self.categorical_features)
        ],
        remainder="drop",
        verbose_feature_names_out=False)

        preprocess_pipeline.set_output(transform="pandas")
        return preprocess_pipeline


In [ ]:
data_preparation = DataPreparation()
#data_preparation.generate_profiling_report(data_ingestion.X, '/content/drive/MyDrive/thesis_data/Results/EDA/data_profiling_report.html')

In [ ]:
# @title
# Data cleaning
features_to_drop = ['message', 'status', 'coming_from_contract', 'Error!', 'gas_price_ratio', 'gas_per_log_event','log_count', 'log_removed', 'log_index', 'length_log', 'normalized_token_transfer','length_from_address','length_transaction_hash']

data_preparation.update_features(features_to_drop)
X_train, X_test, y_train, y_test, groups_train, groups_test = data_ingestion.get_processed_data(dataset_path, grouped_split=True)

X_train.shape

In [ ]:
correlation_save_path = eda_results_path+'feature_correlation_heatmap.png'
ResultsUtils.plot_correlation_heatmap(X_train, save_path = correlation_save_path)

In [ ]:
ResultsUtils.plot_feature_boxplot(X_train, y_train, 'total_gas_cost', save_path= eda_results_path+'feature_boxplot_total_gas_cost.png', log_scale=True )

In [ ]:
ResultsUtils.plot_feature_boxplot(X_train, y_train, 'gas_used_minimum_val', save_path= eda_results_path+'feature_boxplot_gas_used_minimum_val.png', log_scale=True )

In [ ]:
ResultsUtils.plot_feature_boxplot(X_train, y_train, 'value', save_path= eda_results_path+'feature_boxplot_value.png', log_scale=True )

In [ ]:
ResultsUtils.plot_feature_boxplot(X_train, y_train, 'nonce_minimum_val', save_path= eda_results_path+'feature_boxplot_nonce_minimum_val.png', log_scale=False )

In [ ]:
ResultsUtils.plot_feature_boxplot(X_train, y_train, 'values_variance', save_path= eda_results_path+'values_variance.png', log_scale=True )

## Wallet Level Data Leakage Check

In [ ]:
wallet_level_leakage_check = ResultsUtils.get_wallet_level_leakage_check(data_ingestion.X, FeatureTaxanomy.FEATURES_WALLET)
wallet_level_leakage_check.to_csv(experiments_results_path+'wallet_level_leakage_check.csv', index=False)

## Experiments

#####ModelUtils

In [ ]:
class ModelUtils:

  RANDOMFOREST_PARAM_DIST = {
        'rf__n_estimators':      randint(100, 500),
        'rf__max_depth':         [None, 10, 20, 30, 50],
        'rf__min_samples_split': randint(2, 20),
        'rf__min_samples_leaf':  randint(1, 10),
        'rf__max_features':      ['sqrt', 0.3, 0.5],
        'rf__ccp_alpha':         loguniform(1e-5, 1e-2),}

  XGBOOST_PARAM_DIST = {
    'xgb__n_estimators':      randint(100, 600),
    'xgb__max_depth':         randint(3, 11),
    'xgb__learning_rate':     loguniform(0.01, 0.4),
    'xgb__subsample':         uniform(0.6, 0.4),
    'xgb__colsample_bytree':  uniform(0.6, 0.4),
    'xgb__min_child_weight':  randint(1, 10),
    'xgb__reg_lambda':        loguniform(0.1, 10),  # regularization
    'xgb__reg_alpha':         loguniform(0.01, 5),}

  LIGHTGBM_PARAM_DIST = {
    'lgbm__n_estimators':      randint(100, 600),
    'lgbm__num_leaves':        randint(20, 150),
    'lgbm__max_depth':         [-1, 6, 9, 12],
    'lgbm__learning_rate':     loguniform(0.01, 0.3),
    'lgbm__subsample':         uniform(0.6, 0.4),
    'lgbm__subsample_freq':    [1],
    'lgbm__colsample_bytree':  uniform(0.6, 0.4),
    'lgbm__min_child_samples': randint(5, 50),
    'lgbm__reg_lambda':        loguniform(0.1, 10),   # regularization
    'lgbm__reg_alpha':         uniform(0, 5),}

  @staticmethod
  def generate_preprocessing_pipeline(data_preparation):
    return data_preparation.generate_preprocessing_pipeline()

  @staticmethod
  def get_random_forest_pipeline(preprocess_pipeline):
    rf_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
    ('rf', RandomForestClassifier(
         random_state = Protocol.SETTINGS['random_state'],
         n_jobs=1
         ))])

    return rf_pipe


  @staticmethod
  def get_xgboost_pipeline(preprocess_pipeline):
    xgb_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
     ('xgb', XGBClassifier(
         eval_metric='logloss',
         random_state = Protocol.SETTINGS['random_state'],
         n_jobs=1,
         tree_method='hist'
         ))])

    return xgb_pipe


  @staticmethod
  def get_lightgbm_pipeline(preprocess_pipeline):
    lgbm_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
    ('lgbm', LGBMClassifier(
        verbose=1,
        random_state = Protocol.SETTINGS['random_state'],
        n_jobs=1,
        ))])

    return lgbm_pipe


In [ ]:

class ModelTrain:

      def __init__(self, name, pipeline, param_distributions, n_jobs, cv_strategy, scoring="average_precision", n_iter=20, random_state=42):
        self.name = name
        self.pipeline = pipeline
        self.param_distributions = param_distributions
        self.cv_strategy = cv_strategy
        self.scoring = scoring
        self.n_iter = n_iter
        self.random_state = random_state

        self.search = RandomizedSearchCV(
        pipeline,
        param_distributions,
        n_iter= n_iter,
        scoring= scoring,
        refit= 'pr_auc',
        n_jobs= n_jobs,
        cv=cv_strategy,
        error_score='raise',
        verbose=1)


      def fit(self, X, y, groups):
        start = time.time()
        self.search.fit(X, y, groups=groups)
        elapsed = time.time() - start
        self.time = elapsed
        self.cv_summary = ResultsUtils.get_cv_summary(self.search, self.name)

      def fit_random(self, X, y):
        self.search.fit(X, y)
        # TO DO:: Add computational cost/ time to the cv_summary and holdout_summary
        self.cv_summary = ResultsUtils.get_cv_summary(self.search, self.name)

      def evaluate_model(self, X_test, y_test):
        self.holdout_summary, self.y_true, self.y_probs, self.y_pred = ResultsUtils.get_holdout_summary(self.search.best_estimator_, X_test, y_test, self.name)

      def plot_learning_curve(self, X_train, y_train, groups, title, save_path):
        ResultsUtils.plot_learning_curve(self.search.best_estimator_, X_train, y_train, groups, self.cv_strategy, self.scoring['pr_auc'], self.random_state, self.name, save_path)

      def plot_confusion_matrix(self, title, save_path):
        ResultsUtils.plot_confusion_matrix(self.y_true, self.y_pred, title, save_path)


In [ ]:
preprocess_pipeline = data_preparation.generate_preprocessing_pipeline()

In [ ]:
#random forest pipeline
rf_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
    ('rf', RandomForestClassifier(
         random_state = Protocol.SETTINGS['random_state'],
         n_jobs=4
    ))
])

rf_param_distributions = {
    'rf__n_estimators':      randint(100, 500),
    'rf__max_depth':         [None, 10, 20, 30, 50],
    'rf__min_samples_split': randint(2, 20),
    'rf__min_samples_leaf':  randint(1, 10),
    'rf__max_features':      ['sqrt', 0.3, 0.5],
}

set_config(display="diagram")
rf_pipe


In [ ]:
#XGBoost pipeline
xgb_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
     ('xgb', XGBClassifier(
         eval_metric='logloss',
         random_state = Protocol.SETTINGS['random_state'],
         n_jobs=4,
         tree_method='hist'
    ))
])

#XGBoost grid
xgb_param_dist = {
    'xgb__n_estimators':      randint(100, 600),
    'xgb__max_depth':         randint(3, 11),
    'xgb__learning_rate':     loguniform(0.01, 0.4),
    'xgb__subsample':         uniform(0.6, 0.4),
    'xgb__colsample_bytree':  uniform(0.6, 0.4),
    'xgb__min_child_weight':  randint(1, 10)
}

xgb_pipe

In [ ]:
#LightGBM pipeline
lgbm_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
    ('lgbm', LGBMClassifier(
        verbose=1,
        random_state = Protocol.SETTINGS['random_state'],
        n_jobs=1,
    ))
])

lgbm_param_dist = {
    'lgbm__n_estimators':      randint(100, 600),
    'lgbm__num_leaves':        randint(20, 150),
    'lgbm__max_depth':         [-1, 6, 9, 12],
    'lgbm__learning_rate':     loguniform(0.01, 0.3),
    'lgbm__subsample':         uniform(0.6, 0.4),
    'lgbm__subsample_freq':    [1],   #this is above the default of [0] TO DO :: flag this in implementation section and explain
    'lgbm__colsample_bytree':  uniform(0.6, 0.4),
    'lgbm__min_child_samples': randint(5, 50),
}

lgbm_pipe

####Experiment V

#####Grouped Split

In [ ]:
# Grouped Split
rf_model_grouped = ModelTrain("Random Forest", rf_pipe, rf_param_distributions, -1, **Protocol.SETTINGS)
rf_model_grouped.fit(X_train, y_train, groups_train)
rf_model_grouped.cv_summary.to_csv(experiments_results_path+'V_grouped_cv_summary.csv', index=False)

rf_model_grouped.cv_summary.head(1)

#####Random Split

In [ ]:
# Random Split
X_train_random, X_test_random, y_train_random, y_test_random, groups_train_random, groups_test_random = data_ingestion.get_processed_data(dataset_path)
rf_model_random = ModelTrain("Random Forest", rf_pipe, rf_param_distributions, -1, **Protocol.V_SETTINGS_RANDOM)
rf_model_random.fit_random(X_train_random, y_train_random)
rf_model_random.cv_summary.to_csv(experiments_results_path+'V_random_cv_summary.csv', index=False)

rf_model_random.cv_summary.head(1)

####Experiment E1

#####Random Forest

In [ ]:
rf_model = ModelTrain("Random Forest", rf_pipe, rf_param_distributions, -1, **Protocol.SETTINGS)
rf_model.fit(X_train, y_train, groups_train)
rf_model.evaluate_model(X_test, y_test)

In [ ]:
rf_model.cv_summary.head(1)

In [ ]:
rf_model.holdout_summary.head(1)

In [ ]:
rf_model.plot_learning_curve(X_train, y_train, groups_train, "Random Forest", experiments_results_path+'E1_learning_curve_random_forest.png')

In [ ]:
rf_model.plot_confusion_matrix("Random Forest", experiments_results_path+'E1_confusion_matrix_random_forest.png')

#####XGBoost

In [ ]:
xgboost_model = ModelTrain("XGBoost", xgb_pipe, xgb_param_dist, -1, **Protocol.SETTINGS)
xgboost_model.fit(X_train, y_train, groups_train)
xgboost_model.evaluate_model(X_test, y_test)

In [ ]:
xgboost_model.cv_summary.head(1)

In [ ]:
xgboost_model.holdout_summary.head(1)

In [ ]:
xgboost_model.plot_learning_curve(X_train, y_train, groups_train, "XGBoost", experiments_results_path+'E1_learning_curve_xgboost.png')

In [ ]:
xgboost_model.plot_confusion_matrix("XGBoost", experiments_results_path+'E1_confusion_matrix_xgboost.png')

#####LightGBM

In [ ]:
lightgbm_model = ModelTrain("LightGBM", lgbm_pipe, lgbm_param_dist, 2, **Protocol.SETTINGS)
lightgbm_model.fit(X_train, y_train, groups = groups_train)

In [ ]:
lightgbm_model.evaluate_model(X_test, y_test)

In [ ]:
lightgbm_model.cv_summary.head(1)

In [ ]:
lightgbm_model.holdout_summary.head(1)

In [ ]:
lightgbm_model.plot_learning_curve(X_train, y_train, groups_train, "LightGBM", experiments_results_path+'E1_learning_curve_lightgbm.png')

In [ ]:
lightgbm_model.plot_confusion_matrix("LightGBM", experiments_results_path+'E1_confusion_matrix_lightgbm.png')

#####E1 Results

In [ ]:
# Example of how to plot  Overlaied PR curves
results_dict = {
    'Random Forest': (rf_model.y_true, rf_model.y_probs),
    'XGBoost': (xgboost_model.y_true, xgboost_model.y_probs),
    'LightGBM': (lightgbm_model.y_true, lightgbm_model.y_probs),
}

ResultsUtils.plot_pr_curves_overlay(results_dict,'PR Curves', save_path = experiments_results_path+'E1_pr_curves.png')

In [ ]:
E1_cv_summary = ResultsUtils.combine_tables(rf_model.cv_summary, xgboost_model.cv_summary, lightgbm_model.cv_summary)
E1_cv_summary.to_csv(experiments_results_path+'E1_cv_summary.csv', index=False)
E1_cv_summary.head(3)

####Experiment E2

##### SHAP implementation:

In [ ]:
best_pipeline = xgboost_model.search.best_estimator_

#Transform training data so SHAP sees what the model sees
X_train_transformed = best_pipeline.named_steps['preprocess'].transform(X_train)
best_booster = best_pipeline.named_steps['xgb'].get_booster()

#fit the explainer on the transformed data
explainer = shap.TreeExplainer(best_booster)
shap_values = explainer.shap_values(X_train_transformed)

ResultsUtils.plot_shap_beeswarm(shap_values, X_train_transformed, 'XGBoost', experiments_results_path+'E2_SHAP_beeswarm.png' )

In [ ]:

importance_df = ResultsUtils.get_shap_importance_ranking(shap_values, X_train_transformed)

cutoff_percent = 0.20 # keep top 80% and drop bottom 20%
n_features = len(importance_df)
cutoff_index = int(n_features * (1 - cutoff_percent)) #index where the kept features end

ResultsUtils.plot_shap_importance_bar('XGBoost', importance_df, cutoff_index=cutoff_index,
                                          save_path= experiments_results_path+'E2_SHAP_importance.png')

features_to_drop = importance_df.tail(n_features - cutoff_index)['feature'].tolist()

print(f"Total features: {n_features}")
print(f"Keeping top {cutoff_index} features ({(1-cutoff_percent):.0%})")
print(f"Dropping {len(features_to_drop)} features ({cutoff_percent:.0%})")

#####XGBoost on pruned feature set

In [ ]:
data_preparation.update_features(features_to_drop)

X_train, X_test, y_train, y_test, groups_train, groups_test = data_ingestion.get_processed_data(dataset_path, grouped_split=True)

X_train.shape

In [ ]:
preprocess_pipeline = ModelUtils.generate_preprocessing_pipeline(data_preparation)

In [ ]:
#XGBoost pipeline
xgb_pipe = ModelUtils.get_xgboost_pipeline(preprocess_pipeline)

In [ ]:
xgboost_model = ModelTrain("XGBoost", xgb_pipe, ModelUtils.XGBOOST_PARAM_DIST, -1, **Protocol.SETTINGS)
xgboost_model.fit(X_train, y_train, groups_train)
xgboost_model.evaluate_model(X_test, y_test)

In [ ]:
xgboost_model.cv_summary.head(1)

In [ ]:
xgboost_model.holdout_summary.head(1)

In [ ]:
xgboost_model.plot_learning_curve(X_train, y_train, groups_train, "XGBoost", experiments_results_path+'E2_learning_curve_xgboost_pruned.png')

In [ ]:
xgboost_model.plot_confusion_matrix("XGBoost", experiments_results_path+'E2_confusion_matrix_xgboost_pruned.png')

#####Parity Check

In [ ]:
parity_result = ResultsUtils.get_parity_check(
    full_pr_auc=0.970602,
    full_pr_auc_std=0.016195	,
    pruned_pr_auc= xgboost_model.cv_summary['CV PR-AUC (mean)'].iloc[0],
    config_label='SHAP Pruned (20%)'
)

parity_result.to_csv(experiments_results_path+'E2_parity_check.csv', index=False)

parity_result.head(1)

####Experiment E3

In [ ]:

smote_pipe = ImbPipeline(steps=[
    ('preprocess', preprocess_pipeline),
    ('smote', SMOTE(random_state=Protocol.SETTINGS['random_state'])),
    ('xgb', XGBClassifier(
        random_state=Protocol.SETTINGS['random_state'],
        n_jobs=1,
        **{k.replace('xgb__', ''): v for k, v in xgboost_model.search.best_params_.items()}   # frozen champion params
    ))
])

xgboost_smote = ModelTrain("XGBoost", smote_pipe, ModelUtils.XGBOOST_PARAM_DIST, -1, **Protocol.SETTINGS)
xgboost_smote.fit(X_train, y_train, groups_train)
xgboost_smote.evaluate_model(X_test, y_test)

In [ ]:
xgboost_smote.cv_summary.head(1)

In [ ]:
xgboost_smote.holdout_summary.head(1)

In [ ]:
legitimate_count = y_train.value_counts()[0]
fraud_count = y_train.value_counts()[1]
weight_ratio = legitimate_count / fraud_count

print(f"weight_ratio: {weight_ratio:.3f}")

cost_sensitive_pipe = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),
    ('xgb', XGBClassifier(
        random_state=Protocol.SETTINGS['random_state'],
        n_jobs=1,
        scale_pos_weight=weight_ratio,
         **{k.replace('xgb__', ''): v for k, v in xgboost_model.search.best_params_.items()}
    ))
])

xgboost_cost_sensitive = ModelTrain("XGBoost", cost_sensitive_pipe, ModelUtils.XGBOOST_PARAM_DIST, -1, **Protocol.SETTINGS)
xgboost_cost_sensitive.fit(X_train, y_train, groups_train)
xgboost_cost_sensitive.evaluate_model(X_test, y_test)

In [ ]:
xgboost_cost_sensitive.cv_summary.head(1)

In [ ]:
xgboost_cost_sensitive.holdout_summary.head(1)

In [ ]:
#TO DO:: confusion matrix and combined pr curves for baseline, smote and cost sensitive

In [ ]:
#TO DO:: Blind Spot Check -> winning treatment cost sensitive and retest on the full unpruned feature set. compare against the pruned result.